# FlexiSaf Academy AI 'Super Agent'
### Final Project Submission
This notebook demonstrate the implementation of a ***Stateful Hybrid RAG Agent*** designed to handle school using LangGraph and LangChain

In [33]:
import os
import asyncio
from pathlib import Path
from pydantic import SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

# Data Loading and Handling
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Retrieval (The Hybrid Part)
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# AI & Vector Database
from langchain.messages import SystemMessage, HumanMessage
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_cohere import CohereEmbeddings

# Agents & Tools
from langchain.agents import create_agent
from langchain_core.tools import tool

# Memory
from langgraph.checkpoint.memory import InMemorySaver

# ---------------------- CONFIGURATION ---------------------- 
class Config(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env", env_file_encoding="utf-8",
        extra="ignore", env_prefix="", case_sensitive=False
    )
    groq_api_key: SecretStr
    cohere_api_key: SecretStr
try:
    settings = Config()
    llm = ChatGroq(groq_api_key=settings.groq_api_key, model="llama-3.3-70b-versatile", temperature=0.0)
except Exception as e:
    print(f"Failed to load API: {str(e)}")
    settings = None

# 1. RAG Setup: Hybrid Knowledge Base
We use **EnsembleRetriever** combining:
1. **Dense Retrieval:** Cohere v3 Embeddings + ChromaDB(Semantic Understanding)
2. **Sparse Retrieval:** BM25 (Keyword Matching)
Results are fused with **Reciprocal Rank Fusion (RRF)**

In [34]:
def prepare_retriever():
    loader = DirectoryLoader("untitled_folder", glob="**/*.pdf", loader_cls=PyMuPDFLoader, show_progress=True)
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    split_chunks = text_splitter.split_documents(docs)

    try:
        model_embeddings = CohereEmbeddings(model="embed-english-v3.0", cohere_api_key=settings.cohere_api_key.get_secret_value())
        print("✅ Embeddings Initialized Successfuly")
    except Exception as e:
        print(f"❌ Failed to initialize embeddings: {str(e)}")

    db_path = "./db_super_agent"
    if Path(db_path).exists():
        vectorstore = Chroma(
            embedding_function=model_embeddings,
            persist_directory=db_path
        )
        print(f"Loaded existing database from: {db_path}")
    else:
        vectorstore = Chroma.from_documents(
            documents=split_chunks,
            embedding=model_embeddings,
            persist_directory=db_path
        )
        print(f"Vector database created and stored in: {db_path}")

    # Hybrid Set Up
    bm25 = BM25Retriever.from_documents(split_chunks)
    bm25.k = 2
    vs_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

    # This combines them mathematically using Reciprocal Rank Fusion (RRF)
    ensemble = EnsembleRetriever(retrievers=[bm25, vs_retriever], weigth=[0.5, 0.5])

    return ensemble

ensemble_retriever = prepare_retriever()
        

100%|███████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 54.58it/s]

✅ Embeddings Initialized Successfuly
Loaded existing database from: ./db_super_agent


## 2. Agentic Tools & System Prompt
We define a 'Tool' the agent can automatically use when it needs school data. The 'SYSTEM_PROMPT' defines the behavioural guardrails

In [35]:
@tool
def query_school_knowledge(query: str) -> str:
    """Consult this tool for ANY FlexiSaf Academy policies, fees, admissions, calendar or rules"""
    docs = ensemble_retriever.invoke(query)
    return "\n\n".join([d.page_content for d in docs])

SYSTEM_PROMPT = """
You are a conversational and official FlexiSaf Academy AI Assistant.

## RULES
1. Always use the 'query_school_knowledge' tool if the user asks about school details.
2. If the info isn't in the tool, simply say you don't know

## MEMORY BEHAVIOUR
1. Maintain conversational context across all turns.
2. Remember and re-use user provided information (like names, or prebious questions)
3. When asked, provide a clear summary of prior conversation.
4. Ensure continuity in dialogue

## RESPONSE RULES
1. Do not fabricate or hallucinate information
2. Be concise but informative
"""

## 3. State Agent Initialization
We use 'InMemorySaver' to give the agent conversation memory and 'create_agent' to build the decision making brain

In [36]:
memory = InMemorySaver()

# create the stateful agent
rag_agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT,
    tools=[query_school_knowledge],
    checkpointer=memory
)

## 4. Execution Loop
The following block starts the asynchronous chat interface. Type **exit** or **quit** to stop

In [ ]:
async def main():
    config = {"configurable": {"thread_id": "chat-1"}}
    print("\nFlexiSaf Academy 'Super Agent' Online\n")

    while True:
        user_input = input("Ask flexisaf: ")

        if user_input.lower() in ["exit", "quit"]: break

        if not user_input: continue

        print("FlexiSaf: ", end="", flush=True)

        async for chunk in rag_agent.astream(
            {"messages": [HumanMessage(content=user_input)]},
            config=config,
            stream_mode="messages"
        ):
            if hasattr(chunk[0], 'content') and chunk[0].content:
                print(chunk[0].content, end="", flush=True)
        print()
await main()


FlexiSaf Academy 'Super Agent' Online



Ask flexisaf:  who are you?


FlexiSaf: I am a conversational and official FlexiSaf Academy AI Assistant. I can provide information and answer questions about FlexiSaf Academy, and I will do my best to assist you with any queries you may have. Is there something specific you would like to know about the academy?


Ask flexisaf:  whats the school fees


FlexiSaf: Title: FlexiSAF Academy Termly Fee Structure 
Tuition Fees (Per Term): 
●​ Nursery: 120,000 NGN 
●​ Primary (Grade 1-6): 180,000 NGN 
●​ Junior Secondary (JSS 1-3): 250,000 NGN 
●​ Senior Secondary (SSS 1-3): 300,000 NGN 
Mandatory One-Time Fees: 
●​ Registration Fee: 25,000 NGN 
●​ Uniform Set (3 pairs): 30,000 NGN 
Discounts: 
●​ Sibling Discount: 10% off tuition for the third child. 
●​ Early Bird: 5% discount if full session fees are paid before the first week of resumption. 
Note: We operate a strictly cashless policy. Payments must be made via the FlexiSAF Mobile 
App or Direct Bank Transfer.

Title: Rules and Regulations for Students 
Attendance: Students must arrive at the school premises no later than 7:45 AM for the morning 
assembly. 
Uniform Policy: 
●​ Mondays & Tuesdays: Full formal uniform (White shirt, school tie, and blazer). 
●​ Wednesdays: Sportswear/Tracksuit. 
●​ Thursdays: Traditional Attire (To celebrate cultural heritage). 
●​ Fridays: Full formal unif

Ask flexisaf:  What is the school's policy on late coming and the official resumption time?


FlexiSaf: Title: Clubs and Enrichment Programs 
STEM Club: Focuses on Robotics, Coding (Python and Scratch), and 3D printing. 
(Meets Tuesdays).Creative Arts: Music lessons, Drama, and Fine Art. (Meets 
Thursdays). Sports Academy: Football, Basketball, and Swimming. (Meets 
Wednesdays and Fridays). 
FlexiSAF Innovation Lab: A special weekend program for students interested in 
Artificial Intelligence and Software Development. Registration is 15,000 NGN per 
month.

Title: 2026/2027 First Term Calendar 
●​ Resumption Date: Monday, September 14, 2026. 
●​ Orientation Week: September 14 – September 18. 
●​ Mid-Term Break: Wednesday, October 28 – Friday, October 30. 
●​ Parent-Teacher Conference (PTC): Saturday, November 14. 
●​ End of Term Examinations: December 7 – December 11. 
●​ Vacation Date: Friday, December 18, 2026. 
School Hours: 
●​ Mondays to Thursdays: 8:00 AM – 3:30 PM. 
●​ Fridays: 8:00 AM – 1:00 PM (Early closure for Jumu'ah).

Title: FlexiSAF Academy Admission Guidelines 


Ask flexisaf:  can you remember all I've asked you?


FlexiSaf: You've asked me the following questions:

1. Who I am: I told you that I'm a conversational and official FlexiSaf Academy AI Assistant.
2. What the school fees are: I provided you with the fee structure for different levels, including nursery, primary, junior secondary, and senior secondary.
3. The school's policy on late coming and the official resumption time: I informed you that students are expected to arrive at the school premises no later than 7:45 AM for the morning assembly, but the policy on late coming is not explicitly stated.
4. If I can remember all you've asked me: I've just recalled our conversation for you.
